## Preparacion del sistema

In [8]:
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage, SystemMessage
import os
import time

llm= ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4.1",
    temperature=0,
    streaming=True
)

print("Configuracion exitosa")

Configuracion exitosa


## Documentacion

In [ ]:
Base_conocimientos={
    "ayuno": "Se recomienda ayuno de 6 a 8 horas antes de una cirugía.",
    "medicamentos": "Debe consultar con su médico antes de suspender cualquier medicamento.",
    "alergias": "Es importante informar al equipo médico sobre cualquier alergia.",
    "higiene": "Se recomienda bañarse antes del procedimiento según indicaciones médicas.",
    "alcohol": "Evitar consumo de alcohol al menos 24 horas antes de la cirugía.",
    "tiempo": "Se recomienda llegar con aticipasion al centro medico"
}

def buscar_contexto(pregunta):
    contexto = []
    pregunta = pregunta.lower()

    for clave, valor in Base_conocimientos.items():
        if clave in pregunta:
            contexto.append(valor)

        if not contexto:
            contexto.append("Siga siempre las indicaciones generales de su equipo medico antes de una cirugia o examen")
        return contexto
    
print("Documentacion lista")

Documentacion lista


## Codigo general

In [ ]:

def ChatBot_Medico():
    print("==ChatBot preoperatorio==")
    print("Escribe 'salir' para terminar la conversación\\n")

    palabras_prohibidas=["medicacion", "dosis", "diagnostico"]

    while True:
        msg_usuario = input("\\n Tú: ")
        
        if msg_usuario.lower() in ['salir', 'exit', 'quit']:
            print("\\n ¡Hasta luego!")
            break
            
        if not msg_usuario.strip():
            continue
            
        print("\\n ChatBot Medico: ", end="", flush=True)

        try:
            docs_relevantes = buscar_contexto(msg_usuario)
            contexto = "\n".join(docs_relevantes)

            sys_message = f"""Eres un asistente de una clinica el cual esta especilizado en recomendaciones preoperatorias. 
            Tienes que seguir el siguiente reglamento:
            -Solo dar recomendaciones generales.
            -NO puedes recomendar medicamentos ni dosis.
            -NO realizar diagnosticos.
            -Si te piden realizar un diagnostico debes rechazarlo y explicar el pq no debes.

            Usa SOLO la siguiente información como base:
            {contexto}
            """

            messages = [
                {"role": "system", "content": sys_message},
                {"role": "user", "content": msg_usuario}
            ]
            
            lc_messages = [
                SystemMessage(content=sys_message),
                HumanMessage(content=msg_usuario)
            ]
            
            full_response = ""
            respuesta_bloqueada=False
            for chunk in llm.stream(lc_messages):
                content = chunk.content

                if any(p in content.lower() for p in palabras_prohibidas):
                    print("\n\n Evite preguntar sobre medicaciones o diagnosticos")
                    respuesta_bloqueada=True
                    break

                print(content, end="", flush=True)
                full_response += content
                time.sleep(0.02)
                
            print() 
            
        except KeyboardInterrupt:
            print("\\n\\n Interrumpido por el usuario")
            break
        except Exception as e:
            print(f"\\n Error: {e}")
            
    print("\\n¡Gracias por usar el ChatBot de la centro medico tiririlquen!")

ChatBot_Medico()  


==ChatBot preoperatorio==
Escribe 'salir' para terminar la conversación\n


\n ChatBot Medico: ¡Hola! ¿En qué puedo ayudarte hoy? Si tienes alguna pregunta sobre recomendaciones generales antes de una cirugía o examen, estoy aquí para orientarte. Recuerda que siempre es importante seguir las indicaciones de tu equipo médico.
\n ChatBot Medico: Como asistente, solo puedo darte recomendaciones generales. El tiempo de ayuno antes de una cirugía puede variar según el tipo de procedimiento y las indicaciones de tu equipo médico. Es muy importante que sigas las instrucciones específicas que te haya dado tu médico o el personal de la clínica, ya que ellos conocen tu caso particular.

Si tienes dudas sobre el tiempo exacto de ayuno, te recomiendo que consultes directamente con tu equipo médico para asegurarte de cumplir con lo necesario para tu seguridad.
\n ChatBot Medico: No puedo indicarte el tiempo exacto de ayuno, ya que esa información debe ser proporcionada por tu equipo médico según el tipo de cirugía o examen que te realizarán. Te recomiendo seguir siempre la